In [35]:
import pandas as pd

df = pd.read_csv('processed_session_data.csv')

print("Descriptive statistics:")

print(df.describe().round(2))

print("\nCorrelation matrix:")

print(
    df[
        [
            'Mean Temperature (°C)',
            'Mean Humidity (%)',
            'Focus Rating',
            'Mean Reaction Time (ms)'
        ]
    ].corr().round(2)
)

Descriptive statistics:
       Session  Mean Humidity (%)  Mean Temperature (°C)  \
count    30.00              30.00                  30.00   
mean     15.50              55.47                  22.04   
std       8.80               4.22                   0.72   
min       1.00              46.39                  20.42   
25%       8.25              53.18                  21.72   
50%      15.50              55.46                  22.07   
75%      22.75              57.28                  22.38   
max      30.00              67.46                  23.59   

       Min Temperature (°C)  Max Temperature (°C)  Temperature SD  \
count                 30.00                 30.00           30.00   
mean                  21.86                 22.23            0.08   
std                    0.72                  0.71            0.01   
min                   20.17                 20.63            0.06   
25%                   21.52                 21.92            0.07   
50%                  

In [38]:
import plotly.graph_objects as go

# Load the raw simulated data
raw_df = pd.read_csv('simulated_raw_data.csv')

# Convert Timestamp to datetime
raw_df['Timestamp'] = pd.to_datetime(raw_df['Timestamp'])

# Create elapsed time in minutes for each session
raw_df['Time (minutes)'] = (
    raw_df.groupby('Session')['Timestamp']
    .transform(
        lambda x: (x - x.iloc[0]).dt.total_seconds() / 60
    )
)

# Create interactive figure
fig = go.Figure()

# Add one line for each session
for session in sorted(raw_df['Session'].unique()):

    session_data = raw_df[raw_df['Session'] == session]

    fig.add_trace(
        go.Scatter(
            x=session_data['Time (minutes)'],
            y=session_data['Temperature (°C)'],
            mode='lines',
            name=f'Session {session}',
            hovertemplate=(
                'Session: ' + str(session) +
                '<br>Time: %{x:.1f} min' +
                '<br>Temperature: %{y:.2f} °C' +
                '<extra></extra>'
            )
        )
    )

# Configure graph
fig.update_layout(
    title='Temperature Against Time Across 30 Sessions',
    xaxis_title='Time (minutes)',
    yaxis_title='Temperature (°C)',

    xaxis=dict(
        range=[0, 30],
        dtick=5
    ),

    # Click a legend item to show/hide that session
    legend=dict(
        title='Sessions',
        itemclick='toggle',
        itemdoubleclick='toggleothers'
    ),

    hovermode='closest',

    width=1000,
    height=650
)

fig.show()

In [40]:
import plotly.express as px

# Create histogram
fig = px.histogram(
    df,
    x='Mean Temperature (°C)',
    nbins=10,
    title='Distribution of Mean Session Temperature',
    labels={
        'Mean Temperature (°C)': 'Mean Temperature (°C)',
        'count': 'Frequency'
    }
)

# Configure graph
fig.update_layout(
    xaxis_title='Mean Temperature (°C)',
    yaxis_title='Frequency',
    bargap=0.1
)

fig.show()

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare data for regression
x = df[['Temperature (°C)']]
y = df['Focus Rating']

# Create linear regression model
model = LinearRegression()
model.fit(x, y)

# Calculate predictions
df['Predicted Focus'] = model.predict(x)

# Get regression values
slope = model.coef_[0]
intercept = model.intercept_
r_squared = model.score(x, y)

# Create scatter plot
fig = px.scatter(
    df,
    x='Temperature (°C)',
    y='Focus Rating',
    hover_data=[
        'Session',
        'Humidity (%)',
        'Mean Reaction Time (ms)'
    ],
    title='Mean Temperature vs Focus Rating',
    labels={
        'Temperature (°C)': 'Mean Temperature (°C)',
        'Focus Rating': 'Focus Rating'
    }
)

# Add line of best fit
fig.add_scatter(
    x=df['Temperature (°C)'],
    y=df['Predicted Focus'],
    mode='lines',
    name='Line of Best Fit',
    hoverinfo='skip'
)

# Add regression equation and R² to graph
fig.add_annotation(
    x=0.02,
    y=0.98,
    xref='paper',
    yref='paper',
    text=(
        f'y = {slope:.2f}x + {intercept:.2f}'
        f'<br>R² = {r_squared:.2f}'
    ),
    showarrow=False,
    align='left'
)

# Configure graph
fig.update_layout(
    xaxis_title='Mean Temperature (°C)',
    yaxis_title='Focus Rating',
    legend_title='',
    width=900,
    height=600
)

fig.show()

In [ ]:
z = df['Mean Reaction Time (ms)']

model.fit(x, z)

coefficient = model.coef_[0]
r_squared = model.score(x, z)

print(f"Temperature coefficient = {coefficient:.2f} ms/°C")
print(f"R² = {r_squared:.2f}")

Temperature coefficient = -0.67 ms/°C
R² = 0.01


In [ ]:
from scipy.stats import pearsonr

# Pearson correlation
r, p = pearsonr(
    df['Temperature (°C)'],
    df['Focus Rating']
)

print(f"Temperature vs Focus: r = {r:.2f}, p = {p:.2f}")

model.fit(x, y)

coefficient = model.coef_[0]
r_squared = model.score(x, y)

print(f"Temperature coefficient = {coefficient:.2f} focus points/°C")
print(f"R² = {r_squared:.2f}")

Temperature vs Focus: r = 0.06, p = 0.75
Temperature coefficient = 0.12 focus points/°C
R² = 0.00
